# 50 - Evaluation: integrated pilot, main runs, ablation matrix, baselines, injection track, tables (S3.4, S4.2-S4.6)

Thin driver: every computation is a call into `citecurv` (`citecurv.pipeline`, `citecurv.eval.ablation`,
`citecurv.eval.baselines`). Runtime: Colab CPU / high-RAM (the TGN outputs come from `40_tgn.ipynb`, the ORC
values from `30_curvature.ipynb`).

Inputs on Drive (`MyDrive/citecurv/`): `events.parquet`, `works.parquet` (with a top-level `field` column),
`authorships.parquet` (optional), `embeddings.npz`, `event_labels.parquet` (`event_id, label`, L12 rule),
`curvature_cache.parquet` (30_curvature), per seed `runs/TGN_<NAME>_seed<k>/infer/` (40_tgn), and
`configs/split.yaml` from the repository tag (S4.1 cutoff).

Outputs on Drive (`MyDrive/citecurv/eval/`): one folder per run, `runs/<variant>_<seed>/` with `scores.parquet`,
`metrics.json`, `config.json` (resolved config + `config_hash`), `weights.json`, `zstats.json`,
`thresholds.json`; baselines and sanity checks write the same `metrics.json` layout. The results tables are
regenerated from these folders only (`python -m citecurv.eval.ablation --tables`).

Session safety: each run writes its own folder; `run_matrix(..., resume=True)` skips finished cells, so a dead
session loses at most one run (PROVISIONAL(G1): chunk by seed on Colab Pro).

PROVISIONAL: the `no_temporal` cell drops s_temp as well as the TGN memory (`pipeline.no_temporal_drop_s_temp`,
decision D11); labels follow L12 (G3); screen percentiles and ORC radius fall back to `pipeline` defaults while
S2.3 is open. A synthetic dress rehearsal of this whole notebook is in `docs/rehearsal_report.md`.

In [ ]:
# 1. Runtime guard first: the curvature pins need Python 3.12 (GraphRicciCurvature declares scipy<=1.13.1,
#    which has no Python 3.13 wheel). In Colab: Runtime > Change runtime type > runtime version 26.07.
import sys
import platform

print("python", platform.python_version())
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"This notebook needs Python 3.12, found {platform.python_version()}. "
                       "Switch the Colab runtime version, then rerun.")

In [ ]:
# 2. Install citecurv from a release tag plus the stage pins (configs/pins.md). TAG must be set.
TAG = "TAG-PLACEHOLDER"  # release tag of the repository that contains this notebook's code
REPO = "https://github.com/Natboa/Detecting-Network-Citation-Anomalies-Using-Graph-Curvature"
assert TAG != "TAG-PLACEHOLDER", "set TAG to the repository release tag first"
!pip install -q "citecurv @ git+{REPO}@{TAG}" pandas pyarrow pyyaml scikit-learn xgboost
!pip install -q GraphRicciCurvature==0.5.3.2 "scipy<=1.13.1" POT==0.9.6 networkit==11.2.1 networkx
!git clone -q --depth 1 --branch {TAG} {REPO} /content/citecurv-repo  # configs/*.yaml

In [ ]:
# 3. Drive and paths. Values marked SET depend on the run.
from google.colab import drive

drive.mount("/content/drive")
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/citecurv")
EVENTS = DRIVE / "events.parquet"
WORKS = DRIVE / "works.parquet"
AUTHORSHIPS = DRIVE / "authorships.parquet"        # optional (self-citation baseline)
EMBEDDINGS = DRIVE / "embeddings.npz"
LABELS = DRIVE / "event_labels.parquet"            # event_id, label (L12; PROVISIONAL(G3))
CURV_CACHE = DRIVE / "curvature_cache.parquet"     # from 30_curvature (ORC values are reused, not recomputed)
TGN_NAME = "slice"                                 # SET: dataset name used in 40_tgn
CONFIG = Path("/content/citecurv-repo/configs/default.yaml")
SPLIT_YAML = Path("/content/citecurv-repo/configs/split.yaml")  # written by S4.1
EVAL = DRIVE / "eval"                              # runs/<variant>_<seed>/ live below this folder
RESULTS = DRIVE / "eval_results"                   # tables.md + CSV
LEVEL = "real"                                     # tag for the real-data rows of every table

In [ ]:
# 4. Load configuration, split and inputs.
import json

import numpy as np
import pandas as pd
import yaml

from citecurv.config import load_config

cfg = load_config(CONFIG)
SEEDS = cfg["seed_list"]
CUTOFF = int(yaml.safe_load(SPLIT_YAML.read_text())["cutoff_t"])
events = pd.read_parquet(EVENTS)
works = pd.read_parquet(WORKS)
authorships = pd.read_parquet(AUTHORSHIPS) if AUTHORSHIPS.exists() else None
labels = pd.read_parquet(LABELS)
with np.load(EMBEDDINGS, allow_pickle=False) as z:
    embeddings = (z["ids"], z["vectors"])
DATA = dict(events=events, works=works, authorships=authorships, embeddings=embeddings, event_labels=labels,
            cutoff=CUTOFF)
print(len(events), "events |", len(works), "works | cutoff", CUTOFF, "| positives", int(labels["label"].sum()))

In [ ]:
# 5. Reuse the ORC values of the curvature stage, and load the TGN outputs of one seed (40_tgn).
from citecurv.pipeline import orc_memo_from_cache, resolve_config
from citecurv.tgn.infer import load_event_scores

RADIUS = resolve_config(cfg)["curvature"]["orc_radius"]
ORC_MEMO = orc_memo_from_cache(pd.read_parquet(CURV_CACHE), radius=RADIUS) if CURV_CACHE.exists() else {}
print(len(ORC_MEMO), "cached ORC values at radius", RADIUS)


def tgn_inputs(seed):
    # tgn_scores / tgn_embeddings of one seed; {} when 40_tgn has not produced them (s_embed then falls back
    # to static_cosine through embed_mode "auto").
    d = DRIVE / "runs" / f"TGN_{TGN_NAME}_seed{seed}" / "infer"
    if not d.exists():
        print(f"seed {seed}: no TGN outputs at {d}")
        return {}
    with np.load(d / "tgn_event_embeddings.npz") as z:
        emb = {k: z[k] for k in z.files}
    return {"tgn_scores": load_event_scores(d), "tgn_embeddings": emb}


def data_for(seed):
    return {**DATA, **tgn_inputs(seed)}

## S3.4 Integrated pilot and sanity gates

One `full` run (seed 0) and a fresh rerun of it; the gates are written to `pilot_report.json`: D4 fairness (no
(field, year) cell holds more than half of the top-1 % test flags), reproducibility (byte-identical
`scores.parquet`), cold-start consistency, complete top-20 breakdowns.

In [ ]:
from citecurv.eval.ablation import run_dir_name, write_json, write_run
from citecurv.pipeline import (gate_cold_start, gate_fairness, gate_reproducibility, gate_top_breakdown,
                               run_pipeline)

TAGS = {"level": LEVEL, "group": "matrix"}
pilot_dir, rerun_dir = EVAL / "pilot" / "full_0", EVAL / "pilot" / "full_0_rerun"
pilot = run_pipeline(**data_for(0), variant="full", seed=0, cfg=cfg, orc_memo=ORC_MEMO, out_dir=pilot_dir, tags=TAGS)
rerun = run_pipeline(**data_for(0), variant="full", seed=0, cfg=cfg, orc_memo=ORC_MEMO, out_dir=rerun_dir, tags=TAGS)
gates = {
    "d4_fairness": gate_fairness(pilot, works),
    "reproducibility": gate_reproducibility(pilot_dir / "scores.parquet", rerun_dir / "scores.parquet"),
    "cold_start_consistency": gate_cold_start(pilot),
    "top20_breakdown": gate_top_breakdown(pilot, k=20),
}
write_json(RESULTS / "pilot_report.json", {"gates": gates, "test": pilot.metrics["test"], "weights": pilot.weights,
                                           "weights_source": pilot.weights_source,
                                           "config_hash": pilot.config_hash})
print({k: v["status"] for k, v in gates.items()})
print(pilot.metrics["test"], pilot.weights, pilot.weights_source)

## S4.2 / S4.3 Main runs and ablation matrix

Five ablation cells (`full, no_curvature, forman_only, ollivier_only, no_temporal`) and the learned-combiner
cell (L13) for every seed of `seed_list`. ORC values come from the curvature cache (`orc_memo`); cells whose
`metrics.json` exists are skipped (`resume=True`).

In [ ]:
from citecurv.eval.ablation import ABLATION_VARIANTS, EXTRA_CELLS, run_matrix

CELLS = list(ABLATION_VARIANTS) + list(EXTRA_CELLS)
for seed in SEEDS:  # one seed per call, so a dead session resumes at the next unfinished cell
    run_matrix(CELLS, [seed], data_for, EVAL / "matrix" / LEVEL, cfg=cfg, orc_memo=ORC_MEMO, tags=TAGS, resume=True,
               callback=lambda r: print(r.variant, r.seed, r.metrics["test"]["ap"], r.weights_source))

### Timestamp-shuffle check (book 4.2.1)

Test-period timestamps are permuted among test events and the frozen `full` pipeline rescored: thresholds,
z-statistics and weights must come out identical (train/val rows are untouched) and AUROC must move toward 0.5.
The TGN outputs belong to the original order, so this check runs in `static_cosine` mode on both sides unless
`SHUFFLED_TGN` points at a TGN inference on the shuffled stream.

In [ ]:
import copy

from citecurv.pipeline import gate_shuffle, shuffle_test_timestamps

SHUFFLED_TGN = {}  # SET: tgn_inputs-style dict computed on the shuffled stream, if available
cfg_shuffle = copy.deepcopy(cfg)
if not SHUFFLED_TGN:
    cfg_shuffle["scoring"]["components"]["embed_mode"] = "static_cosine"
base = run_pipeline(**DATA, variant="full", seed=0, cfg=cfg_shuffle, orc_memo=ORC_MEMO, tags=TAGS)
pre_memo = {k: v for k, v in ORC_MEMO.items() if k[3] <= CUTOFF}  # pre-cutoff graph state is unchanged
shuffled = run_pipeline(**{**DATA, "events": shuffle_test_timestamps(events, CUTOFF, seed=777), **SHUFFLED_TGN},
                        variant="full", seed=0, cfg=cfg_shuffle, orc_memo=pre_memo, tags=TAGS)
write_run(EVAL / "matrix" / LEVEL / "runs" / run_dir_name("shuffle_full", 0),
          dict(shuffled.metrics, kind="sanity", method="shuffle_full"))
g = gate_shuffle(base, shuffled)
write_run(EVAL / "matrix" / LEVEL / "runs" / run_dir_name("gates", 0),
          {"kind": "sanity", "method": "gates", "seed": 0, "tags": TAGS, "test": None,
           "gates": {**gates, "timestamp_shuffle": g}})
print(g)

## S4.4 Baselines on the same test events

Tier 1 (four rules + rank average), tier 2 (XGBoost on train-period labels), trivial controls (cited-paper age,
reference overlap, random). Every method goes through `evaluate_baseline`, i.e. the same scored test events and
the same metric code as the model. PyGOD (baseline env) and CurvGAD (sealed env) run elsewhere; drop their
`(event_id, score)` parquet files into `EXTERNAL` and they are evaluated here the same way.

In [ ]:
from citecurv.eval.baselines import tier1_stats as t1
from citecurv.eval.baselines import tier2_xgb as t2
from citecurv.pipeline import baseline_record, evaluate_baseline

b = cfg["baselines"]
split = pd.DataFrame({"event_id": pilot.scores["event_id"], "split": pilot.split})
feats = t1.compute_features(events, works, authorships, window_days=b["tier1"]["window_days"],
                            min_venue_citers=b["tier1"]["min_venue_citers"],
                            max_ref_candidates=b["tier1"]["max_ref_candidates"])
ref = feats[feats["event_id"].isin(split.loc[split["split"] == "train", "event_id"])]
test_feats = feats[feats["event_id"].isin(split.loc[split["split"] == "test", "event_id"])]
methods = {"tier1_rank_average": ("baseline", t1.rank_average(test_feats, reference=ref)),
           "ref_overlap": ("control", t1.rule_scores(test_feats, "ref_overlap", reference=ref))}
for c in ("self_citation", "venue_gini", "burst_z"):
    methods[f"tier1_{c}"] = ("baseline", t1.rule_scores(test_feats, c, reference=ref))
x = t2.build_features(events, works, authorships, embeddings, tier1=feats)
methods["tier2_xgb"] = ("baseline", t2.fit_predict(x, labels, split, seed=0, nthread=b["tier2_xgb"]["nthread"],
                                                   num_boost_round=b["tier2_xgb"]["num_boost_round"],
                                                   params={"max_depth": b["tier2_xgb"]["max_depth"],
                                                           "eta": b["tier2_xgb"]["eta"]})[0])
pub = (pd.to_datetime(works.set_index("work_id")["pub_date"]) - pd.Timestamp("1970-01-01")).dt.days
ev_test = events[events["event_id"].isin(split.loc[split["split"] == "test", "event_id"])]
methods["dst_age"] = ("control", pd.DataFrame({"event_id": ev_test["event_id"],
                                               "score": -(ev_test["t"] - pub.reindex(ev_test["dst_id"]).to_numpy())}))
methods["random"] = ("control", pd.DataFrame({"event_id": ev_test["event_id"],
                                              "score": np.random.default_rng(0).random(len(ev_test))}))
EXTERNAL = {"pygod_dominant": DRIVE / "baselines" / "pygod_dominant_seed0.parquet",   # SET when available
            "curvgad": DRIVE / "baselines" / "curvgad_seed0.parquet"}
for name, path in EXTERNAL.items():
    if path.exists():
        methods[name] = ("baseline", pd.read_parquet(path))
for name, (kind, df) in methods.items():
    test = evaluate_baseline(pilot, df, labels)
    write_run(EVAL / "matrix" / LEVEL / "runs" / run_dir_name(name, 0), baseline_record(pilot, kind, name, test, TAGS))
    print(f"{name:22s} AP {test['ap']:.4f} AUROC {test['auroc']:.4f}")

## S4.5 Synthetic cartel injection on the real stream

Cartels are planted after the cutoff only (`t_min = cutoff + 1`), labels are the injected events, and the run
reuses the weights of the main `full` run (calibration off), so no injected event reaches any fitted object.
TGN outputs belong to the clean stream, so the injection track runs with `static_cosine` unless the TGN is
re-inferred on each injected stream. Every number from this section is SYNTHETIC injection on real data and is
labelled so in the tables.

In [ ]:
from citecurv.eval.inject import OBVIOUSNESS, inject_cartel, inject_clique

cfg_inj = copy.deepcopy(cfg)
cfg_inj["scoring"]["weights"] = dict(pilot.weights)
cfg_inj.setdefault("pipeline", {})["calibrate_weights"] = False
cfg_inj["scoring"]["components"]["embed_mode"] = "static_cosine"
exclude = set(labels.loc[labels["label"] == 1, "event_id"].map(events.set_index("event_id")["src_id"]))
for level in [*OBVIOUSNESS, "clique"]:
    if level == "clique":
        inj = inject_clique(events, works, seed=0, t_min=CUTOFF + 1, exclude_ids=exclude, **cfg["inject"]["clique"])
    else:
        inj = inject_cartel(events, works, n_cartels=cfg["inject"]["n_cartels"], seed=0, t_min=CUTOFF + 1,
                            authorships=authorships, exclude_ids=exclude,
                            recipient_share=cfg["inject"]["recipient_share"], **OBVIOUSNESS[level])
    assert (inj.events.set_index("event_id").loc[inj.truth_event_ids, "t"] > CUTOFF).all()
    res = run_pipeline(**{**DATA, "events": inj.events, "event_labels": inj.event_labels()}, variant="full", seed=0,
                       cfg=cfg_inj, orc_memo={k: v for k, v in ORC_MEMO.items() if k[3] <= CUTOFF},
                       out_dir=EVAL / "injection" / f"inj_{level}" / "runs" / run_dir_name("full", 0),
                       tags={"level": f"inj_{level}", "group": "injection_real", "synthetic_injection": True})
    print(level, res.metrics["test"])

## S4.6 Tables, regenerated from the run folders only

Also the label-sensitivity line (L12): every model run's `scores.parquet` re-joined against the broadened label
family (`event_labels_broadened.parquet`, from S0.6), written as group `label_broadened`.

In [ ]:
from citecurv.eval.metrics import evaluate
from citecurv.scoring.aggregate import read_scores

BROAD = DRIVE / "event_labels_broadened.parquet"  # SET when available
if BROAD.exists():
    broad = pd.read_parquet(BROAD)
    for metrics_path in sorted((EVAL / "matrix" / LEVEL / "runs").glob("*/metrics.json")):
        m = json.loads(metrics_path.read_text())
        if m["kind"] != "model" or not (metrics_path.parent / "scores.parquet").exists():
            continue
        s = read_scores(metrics_path.parent / "scores.parquet")
        test = evaluate(s[s["split"] == "test"], broad, fill_missing_label=0)
        write_run(EVAL / "label_broadened" / LEVEL / "runs" / metrics_path.parent.name,
                  {**m, "test": test, "tags": {**m["tags"], "group": "label_broadened"}})
!python -m citecurv.eval.ablation --tables {EVAL} --out {RESULTS} --bundle {RESULTS}/run_bundle.jsonl
print((RESULTS / "tables.md").read_text()[:6000])

## Optional: results explorer for one run

One offline HTML page over the `full` seed-0 run (`citecurv.viz.build_explorer`, docs/explorer.md).

In [ ]:
from IPython.display import HTML

from citecurv.viz import build_explorer, iframe_html

page = build_explorer(RESULTS / "explorer_full_0.html", pilot.scores, components=pilot.components,
                      curvature=pilot.curvature, events=events, works=works, authorships=authorships, labels=labels,
                      metrics=pilot.metrics, weights=pilot.weights, zstats=pilot.zstats.to_dict(), data_note="")
HTML(iframe_html(page))